# 05: Production Monitoring & ML ROI

**Module 3 — Week 14: Monitoring, Alerting, and Quantifying Business Value**

---

## Learning Objectives

1. Understand the four pillars of ML monitoring: system, model, data, and business metrics
2. Build production-grade monitoring dashboards with alerting and SLA tracking
3. Design threshold-based and dynamic alerting systems that avoid alert fatigue
4. Calculate ML ROI using NPV, payback period, and sensitivity analysis
5. Translate model performance into business value (dollars, not AUC)
6. Build stakeholder-facing dashboards with Streamlit
7. Communicate ML results to executives with confidence intervals and honest framing

## Resources

- [Google SRE Book — Monitoring Distributed Systems](https://sre.google/sre-book/monitoring-distributed-systems/)
- [Evidently AI — ML Monitoring](https://www.evidentlyai.com/)
- [NannyML — Post-Deployment Data Science](https://nannyml.com/)
- [Streamlit Documentation](https://docs.streamlit.io/)
- [Made With ML — Monitoring](https://madewithml.com/courses/mlops/monitoring/)
- [Sculley et al. — Hidden Technical Debt in ML Systems](https://papers.nips.cc/paper/2015/hash/86df7dcfd896fcaf2674f757a2463eba-Abstract.html)

In [ ]:
import sys
sys.path.insert(0, '../../..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from datetime import datetime, timedelta

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

np.random.seed(42)
print("All imports loaded successfully.")

---

## Section 1: The Four Pillars of ML Monitoring

Production ML systems require monitoring across **four distinct pillars**. Neglecting any
one of them creates blind spots that lead to silent failures — the most dangerous kind.

### Pillar 1: System Metrics

These are the traditional infrastructure metrics that SREs and DevOps teams already track:

| Metric | What It Measures | Why It Matters |
|--------|-----------------|----------------|
| **Latency (p50/p95/p99)** | Response time distribution | User experience, SLA compliance |
| **Throughput (req/s)** | Request volume | Capacity planning, scaling triggers |
| **Error rate** | Percentage of failed requests | Service reliability |
| **CPU / Memory** | Resource utilization | Cost optimization, OOM prevention |

### Pillar 2: Model Metrics

Model performance degrades over time — this is not a matter of *if* but *when*:

| Metric | What It Measures | Why It Matters |
|--------|-----------------|----------------|
| **Accuracy / AUC** | Predictive quality (when labels available) | Core model health |
| **Prediction distribution** | Shape of output scores | Detects silent model collapse |
| **Calibration** | P(event) vs observed frequency | Trustworthiness of probabilities |

### Pillar 3: Data Metrics

Data is the most common root cause of model failure in production:

| Metric | What It Measures | Why It Matters |
|--------|-----------------|----------------|
| **Feature drift (PSI)** | Distribution shift vs training data | Leading indicator of model decay |
| **Missing values** | Null / NaN rate per feature | Upstream pipeline breakage |
| **Schema violations** | Type errors, unexpected categories | Data contract violations |

### Pillar 4: Business Metrics

Ultimately, the model exists to drive business outcomes:

| Metric | What It Measures | Why It Matters |
|--------|-----------------|----------------|
| **Conversion rate** | Actions taken on predictions | Direct revenue impact |
| **Revenue impact** | Dollars influenced by the model | Executive-level KPI |
| **Customer satisfaction** | NPS, support tickets, churn | Long-term value |

### Why All Four?

| Scenario | System | Model | Data | Business |
|----------|--------|-------|------|----------|
| Server overloaded | Red | Green | Green | Yellow |
| Feature pipeline breaks | Green | Yellow | Red | Yellow |
| Concept drift | Green | Red | Yellow | Red |
| Competitor launches | Green | Green | Green | Red |

Each failure mode lights up a different combination of pillars. Only by monitoring all four
can you diagnose the root cause quickly.

In [ ]:
# ---------------------------------------------------------------------------
# Generate 90 days of simulated production metrics
# ---------------------------------------------------------------------------

n_days = 90
dates = pd.date_range(start="2025-01-01", periods=n_days, freq="D")

# --- Model AUC: stable around 0.92, then gradual degradation from day 60 ---
base_auc = 0.92
auc_noise = np.random.normal(0, 0.005, n_days)
degradation = np.where(
    np.arange(n_days) >= 60,
    -0.003 * (np.arange(n_days) - 60),   # drops ~0.3% per day after day 60
    0
)
model_auc = np.clip(base_auc + auc_noise + degradation, 0.5, 1.0)

# --- Latency (ms): baseline ~80ms p50, spike on day 45 ---
latency_p50 = 80 + np.random.normal(0, 5, n_days)
latency_p95 = latency_p50 * 3.5 + np.random.normal(0, 15, n_days)
latency_p99 = latency_p50 * 6.0 + np.random.normal(0, 25, n_days)

# Inject latency spike on day 45 (± 2 days)
spike_mask = (np.arange(n_days) >= 43) & (np.arange(n_days) <= 47)
latency_p50[spike_mask] *= 3.0
latency_p95[spike_mask] *= 4.0
latency_p99[spike_mask] *= 5.0

# --- Request volume: seasonal weekly pattern + growth ---
day_of_week = np.array([d.weekday() for d in dates])
weekly_pattern = np.where(day_of_week < 5, 1.0, 0.6)  # weekends are 60% of weekdays
base_volume = 10000 + np.arange(n_days) * 50            # gradual growth
request_volume = (base_volume * weekly_pattern + np.random.normal(0, 500, n_days)).astype(int)

# --- Error rate: baseline ~0.5%, elevated during latency spike ---
error_rate = 0.005 + np.random.normal(0, 0.001, n_days)
error_rate[spike_mask] += 0.03  # 3% extra during the spike
error_rate = np.clip(error_rate, 0, 1)

# --- Assemble into DataFrame ---
metrics_df = pd.DataFrame({
    'date': dates,
    'model_auc': model_auc,
    'latency_p50': latency_p50,
    'latency_p95': latency_p95,
    'latency_p99': latency_p99,
    'request_volume': request_volume,
    'error_rate': error_rate,
})

print(f"Production metrics: {len(metrics_df)} days")
print(f"Date range: {metrics_df['date'].min().date()} to {metrics_df['date'].max().date()}")
print()
metrics_df.describe().round(4)

In [ ]:
# Quick sanity check: plot each metric
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].plot(metrics_df['date'], metrics_df['model_auc'], color='steelblue')
axes[0, 0].axhline(y=0.85, color='orange', linestyle='--', label='Warning threshold')
axes[0, 0].set_title('Model AUC Over Time')
axes[0, 0].set_ylabel('AUC')
axes[0, 0].legend()

axes[0, 1].plot(metrics_df['date'], metrics_df['latency_p50'], label='p50')
axes[0, 1].plot(metrics_df['date'], metrics_df['latency_p95'], label='p95')
axes[0, 1].set_title('Latency (ms)')
axes[0, 1].set_ylabel('ms')
axes[0, 1].legend()

axes[1, 0].bar(metrics_df['date'], metrics_df['request_volume'], color='teal', alpha=0.7)
axes[1, 0].set_title('Daily Request Volume')
axes[1, 0].set_ylabel('Requests')

axes[1, 1].plot(metrics_df['date'], metrics_df['error_rate'] * 100, color='crimson')
axes[1, 1].set_title('Error Rate (%)')
axes[1, 1].set_ylabel('%')

for ax in axes.flat:
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.suptitle('Raw Production Metrics — 90 Days', y=1.02, fontsize=14, fontweight='bold')
plt.show()

---

## Section 2: Building a Monitoring Dashboard

A well-designed monitoring dashboard provides **at-a-glance situational awareness**.
Key design principles:

- **KPI cards** at the top: current value, trend direction, status color (green/yellow/red)
- **Time series** in the middle: see trends and anomalies
- **Distributions** at the bottom: understand the shape of your data

### Dashboard Layout Best Practices

1. **Most important metrics** go top-left (the eye starts there)
2. **Consistent time ranges** across all panels
3. **Threshold lines** make it obvious when something is wrong
4. **Color coding**: green = healthy, yellow = warning, red = critical
5. **Annotations** for known events (deploys, incidents, etc.)

In [ ]:
# ---------------------------------------------------------------------------
# Professional 4-Panel Monitoring Dashboard
# ---------------------------------------------------------------------------

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.patch.set_facecolor('#f8f9fa')

# Color palette
HEALTHY = '#28a745'
WARNING = '#ffc107'
CRITICAL = '#dc3545'
PRIMARY = '#007bff'
SECONDARY = '#6c757d'

# --- Panel 1: Model AUC with degradation ---
ax1 = axes[0, 0]
ax1.set_facecolor('#ffffff')
auc_colors = [HEALTHY if v >= 0.88 else WARNING if v >= 0.85 else CRITICAL
              for v in metrics_df['model_auc']]
ax1.scatter(metrics_df['date'], metrics_df['model_auc'], c=auc_colors, s=15, zorder=3)
ax1.plot(metrics_df['date'], metrics_df['model_auc'], color=SECONDARY, alpha=0.3, linewidth=0.8)

# Rolling average
rolling_auc = metrics_df['model_auc'].rolling(7).mean()
ax1.plot(metrics_df['date'], rolling_auc, color=PRIMARY, linewidth=2, label='7-day MA')

ax1.axhline(y=0.88, color=WARNING, linestyle='--', alpha=0.7, label='Warning (0.88)')
ax1.axhline(y=0.85, color=CRITICAL, linestyle='--', alpha=0.7, label='Critical (0.85)')
ax1.set_title('Model AUC', fontsize=13, fontweight='bold', pad=10)
ax1.set_ylabel('AUC Score')
ax1.legend(fontsize=8, loc='lower left')
ax1.set_ylim(0.78, 0.95)

# --- Panel 2: Latency Distribution (p50, p95, p99) ---
ax2 = axes[0, 1]
ax2.set_facecolor('#ffffff')
ax2.fill_between(metrics_df['date'], metrics_df['latency_p50'],
                 metrics_df['latency_p99'], alpha=0.15, color=PRIMARY, label='p50-p99 range')
ax2.fill_between(metrics_df['date'], metrics_df['latency_p50'],
                 metrics_df['latency_p95'], alpha=0.25, color=PRIMARY, label='p50-p95 range')
ax2.plot(metrics_df['date'], metrics_df['latency_p50'], color=HEALTHY, linewidth=1.5, label='p50')
ax2.plot(metrics_df['date'], metrics_df['latency_p95'], color=WARNING, linewidth=1.5, label='p95')
ax2.plot(metrics_df['date'], metrics_df['latency_p99'], color=CRITICAL, linewidth=1.5, label='p99')

# SLA line
ax2.axhline(y=500, color=CRITICAL, linestyle=':', alpha=0.5, label='SLA (p95 < 500ms)')
ax2.set_title('Latency Distribution', fontsize=13, fontweight='bold', pad=10)
ax2.set_ylabel('Latency (ms)')
ax2.legend(fontsize=7, loc='upper left')

# --- Panel 3: Daily Request Volume ---
ax3 = axes[1, 0]
ax3.set_facecolor('#ffffff')
bar_colors = [PRIMARY if v > 6000 else SECONDARY for v in metrics_df['request_volume']]
ax3.bar(metrics_df['date'], metrics_df['request_volume'], color=bar_colors, alpha=0.7, width=0.8)

rolling_vol = metrics_df['request_volume'].rolling(7).mean()
ax3.plot(metrics_df['date'], rolling_vol, color=CRITICAL, linewidth=2, label='7-day MA')
ax3.set_title('Daily Request Volume', fontsize=13, fontweight='bold', pad=10)
ax3.set_ylabel('Requests')
ax3.legend(fontsize=8)

# --- Panel 4: Error Rate with threshold ---
ax4 = axes[1, 1]
ax4.set_facecolor('#ffffff')
err_pct = metrics_df['error_rate'] * 100
err_colors = [CRITICAL if v > 1.0 else WARNING if v > 0.5 else HEALTHY for v in err_pct]
ax4.bar(metrics_df['date'], err_pct, color=err_colors, alpha=0.8, width=0.8)
ax4.axhline(y=1.0, color=CRITICAL, linestyle='--', linewidth=1.5, label='Critical (1%)')
ax4.axhline(y=0.5, color=WARNING, linestyle='--', linewidth=1.5, label='Warning (0.5%)')
ax4.set_title('Error Rate', fontsize=13, fontweight='bold', pad=10)
ax4.set_ylabel('Error Rate (%)')
ax4.legend(fontsize=8)

# Formatting
for ax in axes.flat:
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    ax.tick_params(axis='y', labelsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.suptitle('ML Production Monitoring Dashboard',
             fontsize=16, fontweight='bold', y=1.02)
plt.show()

# Print KPI summary
print("\n" + "="*60)
print("KPI SUMMARY (Last 7 Days)")
print("="*60)
last7 = metrics_df.tail(7)
print(f"  Model AUC:      {last7['model_auc'].mean():.4f}  (trend: {'DOWN' if last7['model_auc'].iloc[-1] < last7['model_auc'].iloc[0] else 'UP'})")
print(f"  Latency p50:    {last7['latency_p50'].mean():.1f} ms")
print(f"  Latency p95:    {last7['latency_p95'].mean():.1f} ms")
print(f"  Request Volume: {last7['request_volume'].mean():.0f} req/day")
print(f"  Error Rate:     {last7['error_rate'].mean()*100:.3f}%")
print("="*60)

---

## Section 3: Alerting and Thresholds

Monitoring without alerting is just *data collection*. But poorly designed alerts
cause **alert fatigue** — the single biggest failure mode of monitoring systems.

### Static vs Dynamic Thresholds

| Approach | Pros | Cons |
|----------|------|------|
| **Static thresholds** | Simple, predictable | Doesn't adapt to trends, high false-positive rate |
| **Dynamic thresholds** (moving avg + std dev) | Adapts to seasonality | More complex, can miss gradual drift |
| **Hybrid** | Best of both worlds | Requires careful tuning |

### Alert Fatigue

If your team receives more than **~5 actionable alerts per on-call shift**, you have a problem:

- Engineers start ignoring alerts
- Real incidents get lost in the noise
- Morale drops, on-call becomes dreaded

**Fix**: Every alert must have a clear **runbook** (what to do when it fires).
If an alert fires and the response is "ignore it," delete that alert.

### Escalation Policies

```
INFO     → Log only, dashboard color change
WARNING  → Slack notification to #ml-alerts
CRITICAL → Page on-call engineer (PagerDuty / OpsGenie)
```

Use **warning** as a heads-up (investigate within hours) and **critical** as
"something is broken right now" (investigate within minutes).

In [ ]:
# ---------------------------------------------------------------------------
# ThresholdMonitor class — static + dynamic alerting
# ---------------------------------------------------------------------------

class ThresholdMonitor:
    """Production-grade threshold monitor with static and dynamic alerting.
    
    Supports:
    - Static thresholds (warning and critical)
    - Dynamic thresholds based on rolling mean +/- k*std
    - Direction-aware: 'above' (e.g., error rate) or 'below' (e.g., AUC)
    """
    
    def __init__(self, metric_name, warning_threshold, critical_threshold,
                 window=7, direction='below'):
        """
        Parameters
        ----------
        metric_name : str
            Human-readable name for the metric.
        warning_threshold : float
            Static threshold for warning alerts.
        critical_threshold : float
            Static threshold for critical alerts.
        window : int
            Rolling window size for dynamic thresholds.
        direction : str
            'below' = alert when value drops below threshold (AUC, accuracy)
            'above' = alert when value exceeds threshold (error rate, latency)
        """
        self.metric_name = metric_name
        self.warning_threshold = warning_threshold
        self.critical_threshold = critical_threshold
        self.window = window
        self.direction = direction
    
    def check(self, values, dates=None):
        """Check a series of values against static and dynamic thresholds.
        
        Parameters
        ----------
        values : array-like
            Metric values to check.
        dates : array-like, optional
            Corresponding dates/timestamps.
        
        Returns
        -------
        list[dict]
            List of alert dicts with keys: severity, timestamp, value, type.
        """
        values = np.array(values)
        if dates is None:
            dates = list(range(len(values)))
        
        alerts = []
        
        # --- Static threshold checks ---
        for i, (val, dt) in enumerate(zip(values, dates)):
            if self.direction == 'below':
                if val < self.critical_threshold:
                    alerts.append({
                        'severity': 'CRITICAL',
                        'timestamp': dt,
                        'value': round(val, 6),
                        'type': 'static',
                        'metric': self.metric_name,
                        'message': f"{self.metric_name} = {val:.4f} < {self.critical_threshold}"
                    })
                elif val < self.warning_threshold:
                    alerts.append({
                        'severity': 'WARNING',
                        'timestamp': dt,
                        'value': round(val, 6),
                        'type': 'static',
                        'metric': self.metric_name,
                        'message': f"{self.metric_name} = {val:.4f} < {self.warning_threshold}"
                    })
            else:  # 'above'
                if val > self.critical_threshold:
                    alerts.append({
                        'severity': 'CRITICAL',
                        'timestamp': dt,
                        'value': round(val, 6),
                        'type': 'static',
                        'metric': self.metric_name,
                        'message': f"{self.metric_name} = {val:.4f} > {self.critical_threshold}"
                    })
                elif val > self.warning_threshold:
                    alerts.append({
                        'severity': 'WARNING',
                        'timestamp': dt,
                        'value': round(val, 6),
                        'type': 'static',
                        'metric': self.metric_name,
                        'message': f"{self.metric_name} = {val:.4f} > {self.warning_threshold}"
                    })
        
        # --- Dynamic threshold checks (rolling mean ± 2 std) ---
        if len(values) > self.window:
            rolling_mean = pd.Series(values).rolling(self.window).mean()
            rolling_std = pd.Series(values).rolling(self.window).std()
            
            for i in range(self.window, len(values)):
                mu = rolling_mean.iloc[i - 1]  # use previous window's stats
                sigma = rolling_std.iloc[i - 1]
                val = values[i]
                dt = dates[i]
                
                if sigma > 0:
                    z_score = abs(val - mu) / sigma
                    if z_score > 3.0:
                        # Only add dynamic alert if no static alert for same point
                        existing = [a for a in alerts
                                    if a['timestamp'] == dt and a['type'] == 'static']
                        if not existing:
                            alerts.append({
                                'severity': 'WARNING',
                                'timestamp': dt,
                                'value': round(val, 6),
                                'type': 'dynamic',
                                'metric': self.metric_name,
                                'message': f"{self.metric_name} anomaly: z-score = {z_score:.1f}"
                            })
        
        return sorted(alerts, key=lambda x: str(x['timestamp']))


print("ThresholdMonitor class defined.")

In [ ]:
# ---------------------------------------------------------------------------
# Run monitors on our simulated data
# ---------------------------------------------------------------------------

# Monitor 1: Model AUC (alert when it drops)
auc_monitor = ThresholdMonitor(
    metric_name='Model AUC',
    warning_threshold=0.88,
    critical_threshold=0.85,
    window=7,
    direction='below'
)

# Monitor 2: Error rate (alert when it rises)
error_monitor = ThresholdMonitor(
    metric_name='Error Rate',
    warning_threshold=0.01,
    critical_threshold=0.02,
    window=7,
    direction='above'
)

# Monitor 3: Latency p95 (alert when it rises)
latency_monitor = ThresholdMonitor(
    metric_name='Latency p95',
    warning_threshold=400,
    critical_threshold=500,
    window=7,
    direction='above'
)

# Run all monitors
auc_alerts = auc_monitor.check(metrics_df['model_auc'].values, metrics_df['date'].values)
error_alerts = error_monitor.check(metrics_df['error_rate'].values, metrics_df['date'].values)
latency_alerts = latency_monitor.check(metrics_df['latency_p95'].values, metrics_df['date'].values)

all_alerts = auc_alerts + error_alerts + latency_alerts

print(f"Total alerts generated: {len(all_alerts)}")
print(f"  AUC alerts:     {len(auc_alerts)} ({sum(1 for a in auc_alerts if a['severity']=='CRITICAL')} critical)")
print(f"  Error alerts:   {len(error_alerts)} ({sum(1 for a in error_alerts if a['severity']=='CRITICAL')} critical)")
print(f"  Latency alerts: {len(latency_alerts)} ({sum(1 for a in latency_alerts if a['severity']=='CRITICAL')} critical)")

print("\n--- Sample Alerts ---")
alerts_df = pd.DataFrame(all_alerts)
if len(alerts_df) > 0:
    alerts_df = alerts_df.sort_values('timestamp')
    print(alerts_df[['severity', 'timestamp', 'metric', 'type', 'message']].head(15).to_string(index=False))

---

## Section 4: SLA Monitoring

**Service Level Agreements (SLAs)** are the contractual promises you make about your
service's reliability. In ML systems, SLAs typically cover:

### Common SLA Targets

| SLA Type | Target | Annual Budget |
|----------|--------|---------------|
| **Latency** (p50) | < 100ms | — |
| **Latency** (p95) | < 500ms | — |
| **Latency** (p99) | < 1,000ms | — |
| **Availability** | 99.9% | 8.76 hours downtime/year |
| **Availability** | 99.99% | 52.6 minutes downtime/year |

### Error Budgets

An **error budget** is the inverse of your SLA target:

- If your SLA is 99.9% availability, your error budget is **0.1%**
- That's 43.8 minutes per month of allowed downtime
- When the budget is exhausted, **freeze all deploys** until next month

This creates a natural tension between shipping features (which burn error budget)
and reliability (which preserves it).

In [ ]:
# ---------------------------------------------------------------------------
# SLA Compliance Calculation
# ---------------------------------------------------------------------------

# Define SLA targets
sla_targets = {
    'latency_p50': {'target': 100, 'direction': 'below', 'label': 'p50 < 100ms'},
    'latency_p95': {'target': 500, 'direction': 'below', 'label': 'p95 < 500ms'},
    'latency_p99': {'target': 1000, 'direction': 'below', 'label': 'p99 < 1000ms'},
    'error_rate':  {'target': 0.01, 'direction': 'below', 'label': 'Error rate < 1%'},
}

print("SLA Compliance Report")
print("=" * 60)

sla_results = {}

for metric, config in sla_targets.items():
    values = metrics_df[metric].values
    if config['direction'] == 'below':
        compliant = values < config['target']
    else:
        compliant = values > config['target']
    
    compliance_pct = compliant.mean() * 100
    violations = (~compliant).sum()
    
    sla_results[metric] = {
        'compliance': compliance_pct,
        'violations': violations,
        'compliant_days': compliant
    }
    
    status = 'PASS' if compliance_pct >= 99.0 else 'FAIL'
    icon = '+' if status == 'PASS' else 'X'
    print(f"  [{icon}] {config['label']:25s} | {compliance_pct:6.1f}% compliant | {violations} violations | {status}")

print("=" * 60)

# Error budget calculation
availability_target = 0.999  # 99.9%
total_minutes_per_month = 30 * 24 * 60  # 43,200
error_budget_minutes = total_minutes_per_month * (1 - availability_target)

# Assume each day with error rate > 1% costs 30 minutes of effective downtime
high_error_days = (metrics_df['error_rate'] > 0.01).sum()
consumed_minutes = high_error_days * 30

print(f"\nError Budget (Monthly):")
print(f"  Total budget:    {error_budget_minutes:.1f} minutes")
print(f"  Consumed:        {consumed_minutes:.1f} minutes")
print(f"  Remaining:       {max(0, error_budget_minutes - consumed_minutes):.1f} minutes")
print(f"  Budget status:   {'HEALTHY' if consumed_minutes < error_budget_minutes else 'EXHAUSTED'}")

In [ ]:
# ---------------------------------------------------------------------------
# SLA Violation Timeline
# ---------------------------------------------------------------------------

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

metrics_to_plot = [
    ('latency_p50', 100, 'p50 Latency (ms)', 'SLA: 100ms'),
    ('latency_p95', 500, 'p95 Latency (ms)', 'SLA: 500ms'),
    ('latency_p99', 1000, 'p99 Latency (ms)', 'SLA: 1000ms'),
    ('error_rate', 0.01, 'Error Rate', 'SLA: 1%'),
]

for ax, (metric, target, ylabel, sla_label) in zip(axes, metrics_to_plot):
    values = metrics_df[metric].values
    violations = values > target
    
    ax.plot(metrics_df['date'], values, color=PRIMARY, linewidth=1)
    ax.axhline(y=target, color=CRITICAL, linestyle='--', linewidth=1.5, label=sla_label)
    
    # Highlight violation days
    if violations.any():
        ax.fill_between(metrics_df['date'], values, target,
                       where=violations, alpha=0.3, color=CRITICAL, label='Violations')
    
    ax.set_ylabel(ylabel, fontsize=9)
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(True, alpha=0.3)

axes[-1].tick_params(axis='x', rotation=30)
plt.suptitle('SLA Violation Timeline', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Monthly SLA Report
# ---------------------------------------------------------------------------

metrics_df['month'] = metrics_df['date'].dt.to_period('M')

monthly_report = []
for month, group in metrics_df.groupby('month'):
    report = {
        'month': str(month),
        'days': len(group),
        'avg_auc': group['model_auc'].mean(),
        'min_auc': group['model_auc'].min(),
        'avg_latency_p95': group['latency_p95'].mean(),
        'max_latency_p95': group['latency_p95'].max(),
        'avg_error_rate_pct': group['error_rate'].mean() * 100,
        'total_requests': group['request_volume'].sum(),
        'p95_sla_compliance': (group['latency_p95'] < 500).mean() * 100,
        'error_sla_compliance': (group['error_rate'] < 0.01).mean() * 100,
    }
    monthly_report.append(report)

monthly_df = pd.DataFrame(monthly_report)
print("Monthly SLA Report")
print("=" * 80)
print(monthly_df.to_string(index=False, float_format='%.2f'))

---

## Section 5: ML ROI Framework

Every ML project must justify its existence in **business terms**. "The model has
0.92 AUC" means nothing to a CFO. "The model saves $2.3M per year in fraud losses"
gets budget approved.

### Cost Categories

| Category | Examples | Typical Range |
|----------|----------|---------------|
| **Infrastructure** | GPU compute, storage, networking, cloud services | $1K–$50K/month |
| **Development** | Engineer time, data labeling, experimentation | $50K–$300K one-time |
| **Maintenance** | Monitoring, retraining, incident response, on-call | $10K–$50K/month |

### Value Categories

| Category | Examples | How to Measure |
|----------|----------|----------------|
| **Revenue increase** | Better recommendations, dynamic pricing, upselling | A/B test vs baseline |
| **Cost reduction** | Automation of manual tasks, fraud prevention | Before/after comparison |
| **Time savings** | Faster decisions, reduced review time | Hours saved x hourly cost |

### Key Formulas

**Net Present Value (NPV):**

$$NPV = \sum_{t=0}^{T} \frac{B_t - C_t}{(1 + r)^t}$$

Where $B_t$ = benefits in period $t$, $C_t$ = costs in period $t$, $r$ = discount rate.

**Payback Period:** The number of periods until cumulative benefits exceed cumulative costs.

**ROI:** $\frac{\text{Total Benefits} - \text{Total Costs}}{\text{Total Costs}} \times 100\%$

In [ ]:
# ---------------------------------------------------------------------------
# ROICalculator class
# ---------------------------------------------------------------------------

class ROICalculator:
    """Calculate ROI, NPV, payback period, and sensitivity for ML projects.
    
    Parameters
    ----------
    costs : dict
        Cost items with monthly values, e.g.,
        {'infrastructure': 5000, 'engineering': 15000, 'maintenance': 3000}
    benefits : dict
        Benefit items with monthly values, e.g.,
        {'revenue_increase': 25000, 'cost_savings': 10000}
    initial_investment : float
        One-time upfront cost (development, setup).
    discount_rate : float
        Annual discount rate (default 10%).
    """
    
    def __init__(self, costs, benefits, initial_investment=0, discount_rate=0.10):
        self.costs = costs
        self.benefits = benefits
        self.initial_investment = initial_investment
        self.discount_rate = discount_rate
        self.monthly_rate = (1 + discount_rate) ** (1/12) - 1
    
    @property
    def monthly_cost(self):
        return sum(self.costs.values())
    
    @property
    def monthly_benefit(self):
        return sum(self.benefits.values())
    
    @property
    def monthly_net(self):
        return self.monthly_benefit - self.monthly_cost
    
    def compute_npv(self, periods=12):
        """Compute Net Present Value over given number of monthly periods."""
        npv = -self.initial_investment  # Period 0: upfront cost
        for t in range(1, periods + 1):
            net_cashflow = self.monthly_benefit - self.monthly_cost
            npv += net_cashflow / (1 + self.monthly_rate) ** t
        return npv
    
    def compute_payback_period(self):
        """Compute payback period in months (when cumulative net > 0)."""
        cumulative = -self.initial_investment
        for t in range(1, 121):  # max 10 years
            cumulative += self.monthly_net
            if cumulative >= 0:
                return t
        return float('inf')  # Never pays back
    
    def cashflow_table(self, periods=12):
        """Generate a month-by-month cashflow table."""
        rows = []
        cumulative = -self.initial_investment
        rows.append({
            'month': 0,
            'costs': self.initial_investment,
            'benefits': 0,
            'net': -self.initial_investment,
            'cumulative': cumulative,
            'npv_cumulative': -self.initial_investment
        })
        
        npv_cum = -self.initial_investment
        for t in range(1, periods + 1):
            net = self.monthly_net
            cumulative += net
            npv_cum += net / (1 + self.monthly_rate) ** t
            rows.append({
                'month': t,
                'costs': self.monthly_cost,
                'benefits': self.monthly_benefit,
                'net': net,
                'cumulative': cumulative,
                'npv_cumulative': npv_cum
            })
        return pd.DataFrame(rows)
    
    def sensitivity_analysis(self, param, range_pct=0.3, steps=10):
        """Analyze how NPV changes when a parameter varies.
        
        Parameters
        ----------
        param : str
            Name of a cost or benefit key to vary.
        range_pct : float
            Percentage range to vary (+/-).
        steps : int
            Number of steps in the range.
        
        Returns
        -------
        pd.DataFrame
            Columns: param_value, npv, roi_pct
        """
        # Find the original value
        if param in self.costs:
            original = self.costs[param]
            is_cost = True
        elif param in self.benefits:
            original = self.benefits[param]
            is_cost = False
        else:
            raise ValueError(f"Parameter '{param}' not found in costs or benefits.")
        
        results = []
        low = original * (1 - range_pct)
        high = original * (1 + range_pct)
        
        for value in np.linspace(low, high, steps):
            # Temporarily modify the parameter
            if is_cost:
                old_val = self.costs[param]
                self.costs[param] = value
            else:
                old_val = self.benefits[param]
                self.benefits[param] = value
            
            npv = self.compute_npv(periods=12)
            total_cost = self.initial_investment + self.monthly_cost * 12
            total_benefit = self.monthly_benefit * 12
            roi = ((total_benefit - total_cost) / total_cost * 100) if total_cost > 0 else 0
            
            results.append({
                'param_value': value,
                'pct_change': (value - original) / original * 100,
                'npv': npv,
                'roi_pct': roi
            })
            
            # Restore original value
            if is_cost:
                self.costs[param] = old_val
            else:
                self.benefits[param] = old_val
        
        return pd.DataFrame(results)
    
    def summary(self):
        """Print a formatted ROI summary."""
        print("ML Project ROI Summary")
        print("=" * 50)
        print(f"\nInitial Investment: ${self.initial_investment:,.0f}")
        print(f"\nMonthly Costs:")
        for k, v in self.costs.items():
            print(f"  {k:25s} ${v:>10,.0f}")
        print(f"  {'TOTAL':25s} ${self.monthly_cost:>10,.0f}")
        print(f"\nMonthly Benefits:")
        for k, v in self.benefits.items():
            print(f"  {k:25s} ${v:>10,.0f}")
        print(f"  {'TOTAL':25s} ${self.monthly_benefit:>10,.0f}")
        print(f"\nMonthly Net:                  ${self.monthly_net:>10,.0f}")
        print(f"12-Month NPV:                 ${self.compute_npv(12):>10,.0f}")
        print(f"Payback Period:               {self.compute_payback_period():>10d} months")
        
        total_cost = self.initial_investment + self.monthly_cost * 12
        total_benefit = self.monthly_benefit * 12
        roi = (total_benefit - total_cost) / total_cost * 100
        print(f"12-Month ROI:                 {roi:>9.1f}%")
        print("=" * 50)


print("ROICalculator class defined.")

In [ ]:
# ---------------------------------------------------------------------------
# Example: ROI for a recommendation engine
# ---------------------------------------------------------------------------

rec_roi = ROICalculator(
    costs={
        'infrastructure': 8000,     # GPU instances, storage
        'engineering': 5000,         # Ongoing feature development (amortized)
        'maintenance': 2000,         # Monitoring, retraining, on-call
    },
    benefits={
        'revenue_increase': 35000,   # Better recommendations → more purchases
        'cost_savings': 5000,        # Less manual curation
    },
    initial_investment=150000,        # 3 months of development
    discount_rate=0.10
)

rec_roi.summary()

# Show cashflow table
print("\nCashflow Table (12 months):")
cf = rec_roi.cashflow_table(12)
print(cf.to_string(index=False, float_format='${:,.0f}'.format))

In [ ]:
# ---------------------------------------------------------------------------
# Sensitivity Analysis
# ---------------------------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sensitivity to revenue increase
sens_revenue = rec_roi.sensitivity_analysis('revenue_increase', range_pct=0.5, steps=20)
axes[0].plot(sens_revenue['pct_change'], sens_revenue['npv'] / 1000, 'b-o', markersize=4)
axes[0].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[0].axvline(x=0, color='gray', linestyle=':', alpha=0.5)
axes[0].set_xlabel('Revenue Change (%)')
axes[0].set_ylabel('12-Month NPV ($K)')
axes[0].set_title('Sensitivity: Revenue Increase', fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Sensitivity to infrastructure cost
sens_infra = rec_roi.sensitivity_analysis('infrastructure', range_pct=0.5, steps=20)
axes[1].plot(sens_infra['pct_change'], sens_infra['npv'] / 1000, 'r-o', markersize=4)
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[1].axvline(x=0, color='gray', linestyle=':', alpha=0.5)
axes[1].set_xlabel('Infrastructure Cost Change (%)')
axes[1].set_ylabel('12-Month NPV ($K)')
axes[1].set_title('Sensitivity: Infrastructure Cost', fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.suptitle('ROI Sensitivity Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.show()

---

## Section 6: ROI Case Study — Churn Model

Let's work through a complete ROI analysis for a **customer churn prediction model**.

### Business Context

| Parameter | Value |
|-----------|-------|
| Active customers | 5,000 |
| Annual churn rate | 25% (1,250 churners/year) |
| Customer Lifetime Value (LTV) | $1,200 |
| Retention offer cost | $50 per targeted customer |

### Without ML Model (Status Quo)

- Send mass retention email to **all** 5,000 customers
- Cost: 5,000 x $50 = **$250,000**
- Saves 15% of churners: 1,250 x 0.15 = 187 customers saved
- Value of saved customers: 187 x $1,200 = **$224,400**
- Net: $224,400 - $250,000 = **-$25,600** (net loss!)

### With ML Model

The model enables **targeted** intervention — only contact customers the model
predicts will churn. The key tradeoff is **precision vs recall**:

- **High recall** (catch more churners) = more offers sent = higher cost
- **High precision** (fewer false positives) = fewer wasted offers = lower cost
- The **optimal threshold** maximizes profit, not F1 score

In [ ]:
# ---------------------------------------------------------------------------
# Churn Model ROI: Full Worked Example
# ---------------------------------------------------------------------------

# Business parameters
N_CUSTOMERS = 5000
CHURN_RATE = 0.25
N_CHURNERS = int(N_CUSTOMERS * CHURN_RATE)   # 1,250
N_NON_CHURNERS = N_CUSTOMERS - N_CHURNERS     # 3,750
OFFER_COST = 50          # Cost per retention offer
CUSTOMER_LTV = 1200      # Value of retaining one customer
RETENTION_RATE = 0.30    # 30% of targeted churners accept the offer and stay

print("Churn Model Business Parameters")
print("=" * 50)
print(f"  Active customers:    {N_CUSTOMERS:,}")
print(f"  Annual churn rate:   {CHURN_RATE:.0%}")
print(f"  Expected churners:   {N_CHURNERS:,}")
print(f"  Offer cost:          ${OFFER_COST:,}")
print(f"  Customer LTV:        ${CUSTOMER_LTV:,}")
print(f"  Retention rate:      {RETENTION_RATE:.0%}")

# --- Baseline: mass email everyone ---
baseline_cost = N_CUSTOMERS * OFFER_COST
baseline_saved = int(N_CHURNERS * 0.15)  # Mass email has lower retention
baseline_value = baseline_saved * CUSTOMER_LTV
baseline_profit = baseline_value - baseline_cost

print(f"\nBaseline (Mass Email to All):")
print(f"  Cost:    ${baseline_cost:>10,}")
print(f"  Saved:   {baseline_saved:>10,} customers")
print(f"  Value:   ${baseline_value:>10,}")
print(f"  Profit:  ${baseline_profit:>10,}")

In [ ]:
# ---------------------------------------------------------------------------
# Cost-Benefit Matrix for Different Operating Points
# ---------------------------------------------------------------------------

# Simulate model operating at different thresholds
# Each threshold gives a different precision/recall tradeoff

operating_points = []

# Simulate a realistic precision-recall curve
thresholds = np.arange(0.1, 0.95, 0.05)
for thresh in thresholds:
    # As threshold increases: precision goes up, recall goes down
    # Using a realistic shape for a decent model (AUC ~ 0.85)
    recall = np.clip(1.0 - 0.9 * thresh**1.5, 0.05, 0.99)
    precision = np.clip(0.25 + 0.65 * thresh**0.8, 0.25, 0.95)
    
    # Calculate business metrics
    true_positives = int(N_CHURNERS * recall)
    predicted_positives = int(true_positives / precision) if precision > 0 else N_CUSTOMERS
    false_positives = predicted_positives - true_positives
    
    # Cost = offers sent to everyone the model flags
    total_offers = predicted_positives
    offer_cost = total_offers * OFFER_COST
    
    # Benefit = churners who accept the offer and stay
    customers_saved = int(true_positives * RETENTION_RATE)
    retention_value = customers_saved * CUSTOMER_LTV
    
    # Profit
    profit = retention_value - offer_cost
    
    operating_points.append({
        'threshold': thresh,
        'precision': precision,
        'recall': recall,
        'true_positives': true_positives,
        'false_positives': false_positives,
        'offers_sent': total_offers,
        'offer_cost': offer_cost,
        'customers_saved': customers_saved,
        'retention_value': retention_value,
        'profit': profit,
        'f1': 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    })

op_df = pd.DataFrame(operating_points)

# Find optimal points
best_profit_idx = op_df['profit'].idxmax()
best_f1_idx = op_df['f1'].idxmax()

print("Cost-Benefit Analysis Across Operating Points")
print("=" * 90)
print(op_df[['threshold', 'precision', 'recall', 'f1', 'offers_sent',
             'offer_cost', 'customers_saved', 'profit']].to_string(
    index=False, float_format='%.3f',
    formatters={'offer_cost': '${:,.0f}'.format, 'profit': '${:,.0f}'.format}
))

print(f"\nBest F1 threshold:    {op_df.loc[best_f1_idx, 'threshold']:.2f} "
      f"(F1={op_df.loc[best_f1_idx, 'f1']:.3f}, Profit=${op_df.loc[best_f1_idx, 'profit']:,.0f})")
print(f"Best Profit threshold: {op_df.loc[best_profit_idx, 'threshold']:.2f} "
      f"(F1={op_df.loc[best_profit_idx, 'f1']:.3f}, Profit=${op_df.loc[best_profit_idx, 'profit']:,.0f})")
print(f"\nBaseline profit (mass email): ${baseline_profit:,.0f}")
print(f"ML model uplift: ${op_df.loc[best_profit_idx, 'profit'] - baseline_profit:,.0f}")

In [ ]:
# ---------------------------------------------------------------------------
# Profit vs Classification Threshold
# ---------------------------------------------------------------------------

fig, ax1 = plt.subplots(figsize=(12, 6))

# Profit curve
color_profit = '#2196F3'
ax1.plot(op_df['threshold'], op_df['profit'] / 1000, 'o-',
         color=color_profit, linewidth=2.5, markersize=6, label='Profit ($K)')
ax1.axhline(y=baseline_profit / 1000, color='gray', linestyle='--',
            linewidth=1.5, label=f'Baseline (${baseline_profit/1000:.0f}K)')
ax1.fill_between(op_df['threshold'], baseline_profit / 1000, op_df['profit'] / 1000,
                 where=op_df['profit'] > baseline_profit,
                 alpha=0.15, color=HEALTHY, label='ML Uplift')

# Mark optimal points
ax1.axvline(x=op_df.loc[best_profit_idx, 'threshold'], color=HEALTHY,
            linestyle=':', linewidth=2, label='Optimal (Profit)')
ax1.axvline(x=op_df.loc[best_f1_idx, 'threshold'], color=WARNING,
            linestyle=':', linewidth=2, label='Optimal (F1)')

ax1.set_xlabel('Classification Threshold', fontsize=12)
ax1.set_ylabel('Profit ($K)', fontsize=12, color=color_profit)
ax1.tick_params(axis='y', labelcolor=color_profit)

# F1 on secondary axis
ax2 = ax1.twinx()
color_f1 = '#FF5722'
ax2.plot(op_df['threshold'], op_df['f1'], 's--',
         color=color_f1, linewidth=1.5, markersize=4, alpha=0.7, label='F1 Score')
ax2.set_ylabel('F1 Score', fontsize=12, color=color_f1)
ax2.tick_params(axis='y', labelcolor=color_f1)

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower left', fontsize=9)

plt.title('Profit vs Classification Threshold\n'
          'The optimal threshold for business value differs from the optimal F1 threshold',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nKey insight: Optimizing for F1 score leaves money on the table.")
print("Always optimize for the business objective when deploying to production.")

---

## Section 7: Streamlit Dashboard Basics

**Streamlit** is the fastest path from a Jupyter notebook to a stakeholder-facing
web dashboard. Key concepts:

| Component | Purpose | Example |
|-----------|---------|--------|
| `st.metric()` | KPI cards with delta | Current AUC, trend arrow |
| `st.pyplot()` / `st.plotly_chart()` | Embed charts | Time series, distributions |
| `st.sidebar` | Controls and filters | Date range, model selector |
| `st.columns()` | Multi-column layout | Side-by-side KPIs |
| `@st.cache_data` | Cache expensive computations | Data loading, model inference |

### Dashboard Architecture

```
+--sidebar--+  +----------main area----------+
|           |  | KPI1 | KPI2 | KPI3 | KPI4   |
| Filters   |  |----------------------------  |
| - dates   |  |  Chart 1    |  Chart 2      |
| - model   |  |----------------------------  |
| - metric  |  |  Chart 3    |  Chart 4      |
+---------- +  +------------------------------+
```

> **Note:** Streamlit cannot run inside Jupyter. The code below is saved as a
> Python string and would be written to `src/dashboard.py`, then run with
> `streamlit run src/dashboard.py`.

In [ ]:
# ---------------------------------------------------------------------------
# Complete Streamlit Dashboard Code
# ---------------------------------------------------------------------------
# This code is meant to be saved as a .py file and run with:
#   streamlit run src/dashboard.py
# It CANNOT run inside Jupyter.

dashboard_code = '''
"""ML Production Monitoring Dashboard — Streamlit App.

Run with:
    streamlit run src/dashboard.py
"""

import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---- Page Config ----
st.set_page_config(
    page_title="ML Monitoring Dashboard",
    page_icon="📊",
    layout="wide"
)

# ---- Data Loading (cached) ----
@st.cache_data
def load_metrics():
    """Generate simulated production metrics."""
    np.random.seed(42)
    n_days = 90
    dates = pd.date_range(start="2025-01-01", periods=n_days, freq="D")

    degradation = np.where(
        np.arange(n_days) >= 60,
        -0.003 * (np.arange(n_days) - 60), 0
    )
    model_auc = np.clip(0.92 + np.random.normal(0, 0.005, n_days) + degradation, 0.5, 1.0)
    latency_p50 = 80 + np.random.normal(0, 5, n_days)
    latency_p95 = latency_p50 * 3.5 + np.random.normal(0, 15, n_days)
    error_rate = np.clip(0.005 + np.random.normal(0, 0.001, n_days), 0, 1)

    day_of_week = np.array([d.weekday() for d in dates])
    weekly = np.where(day_of_week < 5, 1.0, 0.6)
    volume = ((10000 + np.arange(n_days) * 50) * weekly).astype(int)

    return pd.DataFrame({
        "date": dates,
        "model_auc": model_auc,
        "latency_p50": latency_p50,
        "latency_p95": latency_p95,
        "error_rate": error_rate,
        "request_volume": volume,
    })

df = load_metrics()

# ---- Sidebar ----
st.sidebar.title("Controls")
date_range = st.sidebar.date_input(
    "Date range",
    value=(df["date"].min(), df["date"].max()),
)
lookback = st.sidebar.selectbox("Lookback window", [7, 14, 30, 60, 90], index=0)

# Filter data
mask = (df["date"] >= pd.Timestamp(date_range[0])) & (df["date"] <= pd.Timestamp(date_range[1]))
filtered = df[mask]

# ---- KPI Cards ----
st.title("ML Production Monitoring")
last_n = filtered.tail(lookback)
prev_n = filtered.tail(lookback * 2).head(lookback)

col1, col2, col3, col4 = st.columns(4)
col1.metric("Model AUC",
            f"{last_n['model_auc'].mean():.4f}",
            f"{last_n['model_auc'].mean() - prev_n['model_auc'].mean():.4f}")
col2.metric("Latency p95 (ms)",
            f"{last_n['latency_p95'].mean():.0f}",
            f"{last_n['latency_p95'].mean() - prev_n['latency_p95'].mean():.0f}",
            delta_color="inverse")
col3.metric("Error Rate",
            f"{last_n['error_rate'].mean():.3%}",
            f"{(last_n['error_rate'].mean() - prev_n['error_rate'].mean()):.3%}",
            delta_color="inverse")
col4.metric("Daily Requests",
            f"{last_n['request_volume'].mean():,.0f}",
            f"{last_n['request_volume'].mean() - prev_n['request_volume'].mean():,.0f}")

# ---- Charts ----
chart_col1, chart_col2 = st.columns(2)

with chart_col1:
    st.subheader("Model AUC Over Time")
    fig1, ax1 = plt.subplots(figsize=(8, 4))
    ax1.plot(filtered["date"], filtered["model_auc"], color="steelblue")
    ax1.axhline(y=0.85, color="red", linestyle="--", alpha=0.5)
    ax1.set_ylabel("AUC")
    st.pyplot(fig1)

with chart_col2:
    st.subheader("Latency Distribution")
    fig2, ax2 = plt.subplots(figsize=(8, 4))
    ax2.plot(filtered["date"], filtered["latency_p50"], label="p50")
    ax2.plot(filtered["date"], filtered["latency_p95"], label="p95")
    ax2.axhline(y=500, color="red", linestyle="--", alpha=0.5, label="SLA")
    ax2.set_ylabel("ms")
    ax2.legend()
    st.pyplot(fig2)

st.subheader("Daily Request Volume")
st.bar_chart(filtered.set_index("date")["request_volume"])

# ---- Alerts Table ----
st.subheader("Recent Alerts")
alert_data = filtered[filtered["model_auc"] < 0.88][["date", "model_auc"]]
alert_data["severity"] = alert_data["model_auc"].apply(
    lambda x: "CRITICAL" if x < 0.85 else "WARNING"
)
if len(alert_data) > 0:
    st.dataframe(alert_data, use_container_width=True)
else:
    st.success("No active alerts!")
'''

print("Streamlit Dashboard Code")
print("=" * 60)
print(dashboard_code)
print("=" * 60)
print("\nTo run this dashboard:")
print("  1. Save the code above to src/dashboard.py")
print("  2. pip install streamlit")
print("  3. streamlit run src/dashboard.py")
print("\nNote: Cannot run inside Jupyter — Streamlit runs its own web server.")

---

## Section 8: Executive Summary Template

Communicating ML results to non-technical stakeholders requires a fundamentally
different approach than presenting to engineers. Key principles:

### Structure: Problem → Solution → Results → ROI → Next Steps

1. **Problem**: Frame in business terms ("We lose $1.5M/year to customer churn")
2. **Solution**: One sentence on approach ("ML model that predicts churn 30 days in advance")
3. **Results**: Before/after comparison with **confidence intervals**
4. **ROI**: Dollars saved/earned, payback period
5. **Next Steps**: Clear, actionable recommendations

### Honest Reporting

- Always show **confidence intervals**, not just point estimates
- Acknowledge limitations and assumptions
- Use **before vs after** framing — executives understand this intuitively
- Avoid jargon: say "accuracy" not "F1-score", "improvement" not "lift"

In [ ]:
# ---------------------------------------------------------------------------
# Executive Summary Template Generator
# ---------------------------------------------------------------------------

def generate_executive_summary(project_name, problem, solution,
                                before_metric, after_metric, metric_name,
                                confidence_interval, roi_annual,
                                payback_months, next_steps):
    """Generate a markdown executive summary for ML project results."""
    
    improvement = after_metric - before_metric
    improvement_pct = (improvement / before_metric) * 100 if before_metric != 0 else 0
    
    summary = f"""
# Executive Summary: {project_name}

**Date:** {datetime.now().strftime('%B %d, %Y')}
**Status:** Production

---

## Problem
{problem}

## Solution
{solution}

## Results

| Metric | Before | After | Improvement |
|--------|--------|-------|-------------|
| {metric_name} | {before_metric:,.0f} | {after_metric:,.0f} | {improvement:+,.0f} ({improvement_pct:+.1f}%) |

**95% Confidence Interval:** [{confidence_interval[0]:,.0f}, {confidence_interval[1]:,.0f}]

## Return on Investment

- **Annual ROI:** ${roi_annual:,.0f}
- **Payback Period:** {payback_months} months
- **Confidence:** Based on {90} days of production data

## Next Steps

"""
    for i, step in enumerate(next_steps, 1):
        summary += f"{i}. {step}\n"
    
    return summary


# Generate example summary for the churn model
summary = generate_executive_summary(
    project_name="Customer Churn Prediction Model",
    problem="We lose approximately $1.5M annually due to customer churn. "
            "Our current mass-email retention campaign costs $250K but recovers "
            "only $224K — a net loss of $26K.",
    solution="Machine learning model that identifies at-risk customers 30 days "
             "before churn, enabling targeted retention offers to the right "
             "customers at the right time.",
    before_metric=-25600,
    after_metric=op_df.loc[best_profit_idx, 'profit'],
    metric_name="Net Retention Profit",
    confidence_interval=(
        op_df.loc[best_profit_idx, 'profit'] * 0.8,
        op_df.loc[best_profit_idx, 'profit'] * 1.15
    ),
    roi_annual=op_df.loc[best_profit_idx, 'profit'] * 1,  # annual
    payback_months=4,
    next_steps=[
        "Expand model to include engagement features from the mobile app",
        "A/B test personalized retention offers (email vs in-app vs phone)",
        "Set up automated weekly monitoring with Slack alerts",
        "Retrain model quarterly to maintain accuracy",
        "Extend approach to B2B customer segment (est. additional $200K/year)"
    ]
)

print(summary)

In [ ]:
# ---------------------------------------------------------------------------
# Before vs After Visualization with Confidence Bands
# ---------------------------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Panel 1: Before vs After bar chart ---
ax = axes[0]
categories = ['Before ML\n(Mass Email)', 'After ML\n(Targeted)']
values = [baseline_profit, op_df.loc[best_profit_idx, 'profit']]
colors = [CRITICAL if v < 0 else HEALTHY for v in values]

# Confidence intervals for the "after" value
ci_low = op_df.loc[best_profit_idx, 'profit'] * 0.80
ci_high = op_df.loc[best_profit_idx, 'profit'] * 1.15
errors = [[0, values[1] - ci_low], [0, ci_high - values[1]]]

bars = ax.bar(categories, values, color=colors, width=0.5, edgecolor='white', linewidth=2)
ax.errorbar(categories[1], values[1],
            yerr=[[values[1] - ci_low], [ci_high - values[1]]],
            fmt='none', color='black', capsize=10, capthick=2, linewidth=2)

ax.axhline(y=0, color='black', linewidth=0.5)

# Add value labels
for bar, val in zip(bars, values):
    y_pos = val + (2000 if val > 0 else -5000)
    ax.text(bar.get_x() + bar.get_width()/2, y_pos,
            f'${val:,.0f}', ha='center', va='bottom' if val > 0 else 'top',
            fontsize=13, fontweight='bold')

ax.set_ylabel('Annual Profit ($)', fontsize=12)
ax.set_title('Retention Program Profit\nBefore vs After ML Model',
             fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# --- Panel 2: Simulated monthly tracking with confidence band ---
ax2 = axes[1]

months = np.arange(1, 13)
monthly_profit = op_df.loc[best_profit_idx, 'profit'] / 12

# Simulate monthly variation
np.random.seed(123)
monthly_actual = monthly_profit + np.random.normal(0, monthly_profit * 0.15, 12)
cumulative_actual = np.cumsum(monthly_actual)

# Confidence bands (widen over time)
ci_width = np.arange(1, 13) * monthly_profit * 0.08
cumulative_expected = np.cumsum(np.full(12, monthly_profit))

ax2.fill_between(months,
                 cumulative_expected - ci_width,
                 cumulative_expected + ci_width,
                 alpha=0.2, color=PRIMARY, label='95% CI')
ax2.plot(months, cumulative_expected, '--', color=PRIMARY, linewidth=1.5, label='Expected')
ax2.plot(months, cumulative_actual, 'o-', color=HEALTHY, linewidth=2, markersize=6, label='Actual')

ax2.set_xlabel('Month', fontsize=12)
ax2.set_ylabel('Cumulative Profit ($)', fontsize=12)
ax2.set_title('Cumulative Profit Tracking\nWith Confidence Bands', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_xticks(months)

plt.tight_layout()
plt.show()

---

## Section 9: Exercises

Complete the following exercises to reinforce the concepts from this notebook.
Each exercise has a TODO cell with hints, followed by a commented-out solution.

### Exercise 1: Add Feature Drift (PSI) to the Monitoring Dashboard

**Population Stability Index (PSI)** measures how much a feature distribution has
shifted from the training baseline. The formula is:

$$PSI = \sum_{i=1}^{B} (p_i^{\text{actual}} - p_i^{\text{expected}}) \cdot \ln\left(\frac{p_i^{\text{actual}}}{p_i^{\text{expected}}}\right)$$

Where $B$ is the number of bins, $p_i^{\text{actual}}$ is the proportion in bin $i$
for the production data, and $p_i^{\text{expected}}$ is the proportion for training data.

**Interpretation:**
- PSI < 0.1 → No significant shift
- 0.1 <= PSI < 0.25 → Moderate shift (investigate)
- PSI >= 0.25 → Significant shift (retrain)

**Task:** 
1. Implement a `compute_psi()` function
2. Simulate 90 days of PSI values with drift starting at day 50
3. Add a 5th panel to the monitoring dashboard showing PSI over time

In [ ]:
# TODO: Exercise 1 — Feature Drift PSI

# Step 1: Implement compute_psi
def compute_psi(expected, actual, bins=10):
    """Compute Population Stability Index between two distributions.
    
    Parameters
    ----------
    expected : array-like
        Reference (training) distribution.
    actual : array-like
        Production distribution.
    bins : int
        Number of bins.
    
    Returns
    -------
    float
        PSI value.
    """
    # YOUR CODE HERE
    pass

# Step 2: Simulate 90 days of PSI values
# Hint: Use np.random.normal with gradually increasing mean shift after day 50
# daily_psi = ...

# Step 3: Create a 5-panel dashboard (add PSI as the 5th panel)
# Hint: Use plt.subplots(3, 2, ...) and leave one panel empty or use it for a summary

In [ ]:
# --- Solution: Exercise 1 ---

# def compute_psi(expected, actual, bins=10):
#     """Compute Population Stability Index."""
#     # Create bins from the expected distribution
#     breakpoints = np.percentile(expected, np.linspace(0, 100, bins + 1))
#     breakpoints[0] = -np.inf
#     breakpoints[-1] = np.inf
#     
#     # Count proportions in each bin
#     expected_counts = np.histogram(expected, bins=breakpoints)[0]
#     actual_counts = np.histogram(actual, bins=breakpoints)[0]
#     
#     # Convert to proportions (with small epsilon to avoid division by zero)
#     eps = 1e-6
#     expected_pct = expected_counts / len(expected) + eps
#     actual_pct = actual_counts / len(actual) + eps
#     
#     # PSI formula
#     psi = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))
#     return psi
#
# # Simulate daily PSI with drift
# np.random.seed(42)
# training_data = np.random.normal(0, 1, 10000)
# daily_psi = []
# for day in range(n_days):
#     if day < 50:
#         shift = 0
#     else:
#         shift = 0.02 * (day - 50)  # Gradual drift
#     production_sample = np.random.normal(shift, 1 + shift * 0.1, 1000)
#     psi_val = compute_psi(training_data, production_sample)
#     daily_psi.append(psi_val)
#
# daily_psi = np.array(daily_psi)
#
# # 5-Panel Dashboard
# fig, axes = plt.subplots(3, 2, figsize=(16, 12))
#
# # Panel 1-4: same as before
# axes[0, 0].plot(dates, model_auc, color='steelblue')
# axes[0, 0].axhline(y=0.85, color='red', linestyle='--')
# axes[0, 0].set_title('Model AUC')
#
# axes[0, 1].plot(dates, latency_p50, label='p50')
# axes[0, 1].plot(dates, latency_p95, label='p95')
# axes[0, 1].set_title('Latency')
# axes[0, 1].legend()
#
# axes[1, 0].bar(dates, request_volume, color='teal', alpha=0.7)
# axes[1, 0].set_title('Request Volume')
#
# axes[1, 1].plot(dates, error_rate * 100, color='crimson')
# axes[1, 1].set_title('Error Rate (%)')
#
# # Panel 5: Feature Drift PSI
# psi_colors = ['green' if v < 0.1 else 'orange' if v < 0.25 else 'red' for v in daily_psi]
# axes[2, 0].bar(dates, daily_psi, color=psi_colors, alpha=0.8)
# axes[2, 0].axhline(y=0.1, color='orange', linestyle='--', label='Moderate (0.1)')
# axes[2, 0].axhline(y=0.25, color='red', linestyle='--', label='Significant (0.25)')
# axes[2, 0].set_title('Feature Drift (PSI)')
# axes[2, 0].legend()
#
# # Panel 6: Summary
# axes[2, 1].axis('off')
# summary_text = (
#     f"Dashboard Summary\n\n"
#     f"Model AUC (latest): {model_auc[-1]:.4f}\n"
#     f"Latency p95 (avg):  {np.mean(latency_p95):.0f} ms\n"
#     f"Error Rate (avg):   {np.mean(error_rate)*100:.3f}%\n"
#     f"Feature PSI (latest): {daily_psi[-1]:.4f}\n"
#     f"Days with PSI > 0.25: {sum(daily_psi > 0.25)}"
# )
# axes[2, 1].text(0.1, 0.5, summary_text, transform=axes[2, 1].transAxes,
#                 fontsize=12, verticalalignment='center', fontfamily='monospace',
#                 bbox=dict(boxstyle='round', facecolor='lightyellow'))
#
# plt.tight_layout()
# plt.suptitle('5-Pillar ML Monitoring Dashboard', fontsize=14, fontweight='bold', y=1.01)
# plt.show()

### Exercise 2: Calculate ROI for a Fraud Detection Model

**Scenario:**
- Monthly transaction volume: 1,000,000 transactions
- Average transaction value: $150
- Fraud rate: 0.5% (5,000 fraudulent transactions/month)
- Current rule-based system catches 60% of fraud
- ML model catches 85% of fraud (from offline evaluation)
- False positive rate (current): 2% → each false positive costs $10 in manual review
- False positive rate (ML model): 0.5%
- ML infrastructure cost: $12,000/month
- Initial development cost: $200,000

**Task:** Use the `ROICalculator` to compute the ROI, payback period, and run
sensitivity analysis on the fraud catch rate.

In [ ]:
# TODO: Exercise 2 — Fraud Detection ROI

# Step 1: Calculate the monthly value of the current system vs the ML model
# Current system:
#   - Fraud caught: 5000 * 0.60 = 3000 transactions * $150 = $450,000 saved
#   - False positive cost: 1,000,000 * 0.02 * $10 = $200,000
#   - Net value: $250,000/month

# ML model:
#   - Fraud caught: 5000 * 0.85 = ? saved
#   - False positive cost: 1,000,000 * 0.005 * $10 = ?
#   - Net value: ?

# Step 2: The INCREMENTAL benefit of ML = ML net value - current net value
# incremental_benefit = ...

# Step 3: Create ROICalculator and compute summary
# fraud_roi = ROICalculator(
#     costs={...},
#     benefits={...},
#     initial_investment=...,
# )
# fraud_roi.summary()

# Step 4: Sensitivity analysis on fraud detection improvement
# What if the ML model only catches 75% instead of 85%?

In [ ]:
# --- Solution: Exercise 2 ---

# MONTHLY_TRANSACTIONS = 1_000_000
# AVG_TXN_VALUE = 150
# FRAUD_RATE = 0.005
# MONTHLY_FRAUD = int(MONTHLY_TRANSACTIONS * FRAUD_RATE)  # 5,000
# REVIEW_COST = 10  # per false positive
#
# # Current system
# current_catch_rate = 0.60
# current_fpr = 0.02
# current_saved = MONTHLY_FRAUD * current_catch_rate * AVG_TXN_VALUE  # $450,000
# current_fp_cost = MONTHLY_TRANSACTIONS * current_fpr * REVIEW_COST  # $200,000
# current_net = current_saved - current_fp_cost  # $250,000
#
# # ML model
# ml_catch_rate = 0.85
# ml_fpr = 0.005
# ml_saved = MONTHLY_FRAUD * ml_catch_rate * AVG_TXN_VALUE  # $637,500
# ml_fp_cost = MONTHLY_TRANSACTIONS * ml_fpr * REVIEW_COST  # $50,000
# ml_net = ml_saved - ml_fp_cost  # $587,500
#
# incremental_benefit = ml_net - current_net  # $337,500/month
#
# print(f"Current system net value: ${current_net:,.0f}/month")
# print(f"ML model net value:       ${ml_net:,.0f}/month")
# print(f"Incremental benefit:      ${incremental_benefit:,.0f}/month")
#
# fraud_roi = ROICalculator(
#     costs={
#         'infrastructure': 12000,
#         'maintenance': 5000,
#     },
#     benefits={
#         'additional_fraud_caught': ml_saved - current_saved,
#         'reduced_fp_reviews': current_fp_cost - ml_fp_cost,
#     },
#     initial_investment=200000,
#     discount_rate=0.10
# )
#
# fraud_roi.summary()
#
# # Sensitivity to fraud catch improvement
# sens = fraud_roi.sensitivity_analysis('additional_fraud_caught', range_pct=0.4, steps=15)
# plt.figure(figsize=(10, 5))
# plt.plot(sens['pct_change'], sens['npv'] / 1000, 'o-', color='steelblue')
# plt.axhline(y=0, color='red', linestyle='--')
# plt.xlabel('Change in Fraud Detection Value (%)')
# plt.ylabel('12-Month NPV ($K)')
# plt.title('Sensitivity: Fraud Detection ROI', fontweight='bold')
# plt.grid(True, alpha=0.3)
# plt.tight_layout()
# plt.show()

### Exercise 3: Design Alerting Rules for a Recommendation System

**Scenario:** You run a product recommendation system for an e-commerce site.

**Task:** Design alerting rules across all four monitoring pillars.
For each alert, specify:
- Metric name
- Warning threshold
- Critical threshold
- Expected response (runbook)

Then implement the monitors using the `ThresholdMonitor` class and generate
simulated data to test them.

In [ ]:
# TODO: Exercise 3 — Recommendation System Alerting

# Step 1: Define alerting rules (fill in the table)
# alerting_rules = {
#     'system': [
#         {'metric': 'inference_latency_p95', 'warning': ???, 'critical': ???,
#          'direction': 'above', 'runbook': 'Scale up serving instances'},
#     ],
#     'model': [
#         {'metric': 'click_through_rate', 'warning': ???, 'critical': ???,
#          'direction': 'below', 'runbook': '???'},
#     ],
#     'data': [
#         # What data quality issues could affect recommendations?
#     ],
#     'business': [
#         # What business metrics would indicate the rec system is failing?
#     ],
# }

# Step 2: Simulate 60 days of recommendation system metrics
# Include at least one anomaly in each pillar

# Step 3: Create ThresholdMonitor instances and run checks
# Print the alert summary

In [ ]:
# --- Solution: Exercise 3 ---

# np.random.seed(42)
# n = 60
# rec_dates = pd.date_range('2025-01-01', periods=n, freq='D')
#
# # Simulate metrics
# # System: inference latency (ms)
# rec_latency = 50 + np.random.normal(0, 5, n)
# rec_latency[30:35] *= 4  # Spike during a deployment
#
# # Model: click-through rate (CTR)
# rec_ctr = 0.08 + np.random.normal(0, 0.005, n)
# rec_ctr[45:] -= 0.02  # Drop after catalog update
#
# # Data: missing product features rate
# rec_missing = 0.01 + np.abs(np.random.normal(0, 0.003, n))
# rec_missing[40:50] = 0.15  # Catalog feed issue
#
# # Business: revenue per session
# rec_revenue = 2.50 + np.random.normal(0, 0.2, n)
# rec_revenue[45:] -= 0.5  # Drops with CTR
#
# # Define monitors
# monitors = [
#     ThresholdMonitor('Rec Latency p95', warning_threshold=100,
#                      critical_threshold=200, direction='above'),
#     ThresholdMonitor('Click-Through Rate', warning_threshold=0.06,
#                      critical_threshold=0.04, direction='below'),
#     ThresholdMonitor('Missing Features Rate', warning_threshold=0.05,
#                      critical_threshold=0.10, direction='above'),
#     ThresholdMonitor('Revenue/Session', warning_threshold=2.0,
#                      critical_threshold=1.5, direction='below'),
# ]
#
# data_series = [rec_latency, rec_ctr, rec_missing, rec_revenue]
#
# print("Recommendation System Alert Summary")
# print("=" * 60)
# all_rec_alerts = []
# for monitor, data in zip(monitors, data_series):
#     alerts = monitor.check(data, rec_dates)
#     all_rec_alerts.extend(alerts)
#     critical = sum(1 for a in alerts if a['severity'] == 'CRITICAL')
#     warning = sum(1 for a in alerts if a['severity'] == 'WARNING')
#     print(f"  {monitor.metric_name:25s} | {warning} warnings, {critical} critical")
#
# print(f"\nTotal alerts: {len(all_rec_alerts)}")
#
# # Plot all four metrics
# fig, axes = plt.subplots(2, 2, figsize=(14, 8))
# titles = ['Inference Latency (ms)', 'Click-Through Rate', 'Missing Features Rate', 'Revenue/Session ($)']
# for ax, data, title in zip(axes.flat, data_series, titles):
#     ax.plot(rec_dates, data, 'o-', markersize=3)
#     ax.set_title(title, fontweight='bold')
#     ax.tick_params(axis='x', rotation=30)
#     ax.grid(True, alpha=0.3)
# plt.tight_layout()
# plt.suptitle('Recommendation System Monitoring', fontsize=14, fontweight='bold', y=1.02)
# plt.show()

---

## Section 10: Key Takeaways

### Monitor All Four Pillars

System metrics alone are not enough. You need **system, model, data, and business**
metrics to detect all failure modes. A model can return 200 OK while producing
garbage predictions.

### Alert on Business Metrics, Not Just Model Metrics

A drop in AUC from 0.92 to 0.90 might be fine. A drop in conversion rate from
3.5% to 2.8% is a revenue emergency. Business metrics are the ultimate ground truth.

### Always Quantify ROI Before and After Deployment

Before building: estimate ROI to justify the project. After deployment: track
actual ROI to prove value and secure continued investment. The ROI framework
(NPV, payback period, sensitivity analysis) is your best tool.

### Speak the Language of the Business

Translate everything into dollars:
- "AUC improved by 0.03" → "We catch 15% more fraud, saving $180K/month"
- "Latency p95 is 450ms" → "5% of customers wait >450ms, costing us $X in abandonment"
- "Feature drift PSI is 0.3" → "The model needs retraining or we risk losing $Y/week"

### Streamlit Is the Fastest Path to Stakeholder Dashboards

You can go from a Jupyter notebook to a shareable web dashboard in under an hour.
Use `st.metric()` for KPIs, `st.pyplot()` for charts, and `st.sidebar` for
interactive controls.

### What's Next

Apply everything you have learned in **Module 3 projects**:
- Build an end-to-end ML pipeline with monitoring
- Deploy a model with SLA tracking and alerting
- Present ROI analysis to a (simulated) executive audience
- Create a Streamlit dashboard for stakeholder consumption

---

*Notebook complete. You now have the tools to monitor ML systems in production
and communicate their value in the language that matters: business impact.*